# Análise de Tokenização — multilingual-e5-large
## Vade Mecum EC134/2024

O modelo `intfloat/multilingual-e5-large` tem **limite de 512 tokens** por sequência.

O pipeline prefixia cada chunk com `"passage: "` antes de gerar o embedding:
```python
textos_prefixados = [f"passage: {t}" for t in textos]
```

Este notebook mede quantos tokens cada chunk ocupa **já com o prefixo**, mostra quais são truncados e o que se perde.

| Passo | Descrição |
|-------|-----------|
| 1 | Extrair e chunkar o PDF |
| 2 | Carregar o tokenizador do e5-large |
| 3 | Contar tokens por chunk (com prefixo) |
| 4 | Estatísticas e distribuição |
| 5 | Análise do conteúdo truncado |
| 6 | Piores casos |
| 7 | Conclusão e recomendações |


In [ ]:
import sys
import statistics
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

CAMINHO_PDF = ROOT / 'data' / 'Vade_mecum_EC134_2024.pdf'
MAX_TOKENS = 512  # limite do modelo
PREFIXO = 'passage: '  # adicionado em GeradorEmbeddings.gerar_batch()

assert CAMINHO_PDF.exists(), f'PDF não encontrado: {CAMINHO_PDF}'
print(f'✔  {CAMINHO_PDF.name}  ({CAMINHO_PDF.stat().st_size / 1e6:.1f} MB)')

## Passo 1 — Extrair e chunkar o PDF

Mesmo algoritmo usado nos benchmarks.

In [ ]:
from ana.rag.ingestao import extrair_texto_pdf, processar_documento
from ana.rag.modelos import TipoDocumento, VigenciaStatus

texto = extrair_texto_pdf(CAMINHO_PDF)
chunks = processar_documento(
    texto=texto,
    fonte='Vade Mecum EC134/2024',
    tipo=TipoDocumento.LEI_FEDERAL,
    vigencia=VigenciaStatus.ATIVA,
)

print(f'✔  {len(chunks):,} chunks gerados')
tamanhos_chars = [len(c.texto) for c in chunks]
print(f'   chars — média={statistics.mean(tamanhos_chars):.0f}  mediana={statistics.median(tamanhos_chars):.0f}  max={max(tamanhos_chars):,}')

## Passo 2 — Carregar o tokenizador

Usa o mesmo tokenizador do modelo. O e5-large é baseado no **XLM-RoBERTa**, que inclui tokens especiais `[CLS]` e `[SEP]` dentro dos 512.

In [ ]:
from transformers import AutoTokenizer
from ana.config_modelos import obter_modelos

nome_modelo = obter_modelos().ativo.embeddings.modelo
print(f'Carregando tokenizador: {nome_modelo} ...')
tokenizer = AutoTokenizer.from_pretrained(nome_modelo)
print(f'✔  Tokenizador carregado')
print(f'   Vocab size     : {tokenizer.vocab_size:,}')
print(f'   Max pos embeds : {tokenizer.model_max_length}')
print()

# Quantos tokens o prefixo consome?
n_tokens_prefixo = len(tokenizer(PREFIXO, add_special_tokens=False)['input_ids'])
print(f'Prefixo "{PREFIXO}" ocupa {n_tokens_prefixo} tokens')
print(f'Tokens disponíveis para o texto: {MAX_TOKENS} - 2 (CLS+SEP) - {n_tokens_prefixo} (prefixo) = {MAX_TOKENS - 2 - n_tokens_prefixo}')

## Passo 3 — Contar tokens por chunk

Tokeniza `"passage: {texto}"` para cada chunk — exatamente o que o modelo recebe.

In [ ]:
registros = []

for chunk in chunks:
    texto_prefixado = PREFIXO + chunk.texto

    # Tokeniza SEM truncamento para ver o tamanho real
    ids_completos = tokenizer(
        texto_prefixado,
        add_special_tokens=True,
        truncation=False,
    )['input_ids']

    n_total = len(ids_completos)
    truncado = n_total > MAX_TOKENS
    tokens_perdidos = max(0, n_total - MAX_TOKENS)

    # Texto que o modelo NÃO vê (decodifica os tokens além do limite)
    if truncado:
        texto_perdido = tokenizer.decode(
            ids_completos[MAX_TOKENS:],
            skip_special_tokens=True,
        )
        pct_perdida = tokens_perdidos / n_total * 100
    else:
        texto_perdido = ''
        pct_perdida = 0.0

    registros.append({
        'artigo': chunk.metadata.artigo,
        'chars': len(chunk.texto),
        'tokens_total': n_total,
        'tokens_perdidos': tokens_perdidos,
        'pct_perdida': round(pct_perdida, 1),
        'truncado': truncado,
        'texto_perdido': texto_perdido,
        'texto': chunk.texto,
    })

df = pd.DataFrame(registros)
print(f'✔  {len(df):,} chunks tokenizados')

## Passo 4 — Estatísticas e distribuição

In [ ]:
total = len(df)
truncados = df['truncado'].sum()
ok = total - truncados

print('══════════════════════════════════════════')
print('  Estatísticas de tokenização')
print('══════════════════════════════════════════')
print(f'  Total de chunks          : {total:>7,}')
print(f'  Dentro do limite (≤ 512) : {ok:>7,}  ({ok/total*100:.1f}%)')
print(f'  Truncados    (> 512)     : {truncados:>7,}  ({truncados/total*100:.1f}%)')
print()
print('  Distribuição de tokens por chunk:')
print(f'    média   : {df["tokens_total"].mean():.1f}')
print(f'    mediana : {df["tokens_total"].median():.0f}')
print(f'    p75     : {df["tokens_total"].quantile(0.75):.0f}')
print(f'    p90     : {df["tokens_total"].quantile(0.90):.0f}')
print(f'    p95     : {df["tokens_total"].quantile(0.95):.0f}')
print(f'    p99     : {df["tokens_total"].quantile(0.99):.0f}')
print(f'    max     : {df["tokens_total"].max()}')
print()
if truncados > 0:
    trunc = df[df['truncado']]
    print('  Nos chunks truncados:')
    print(f'    tokens perdidos — média : {trunc["tokens_perdidos"].mean():.1f}')
    print(f'    tokens perdidos — max   : {trunc["tokens_perdidos"].max()}')
    print(f'    % do texto perdido — média : {trunc["pct_perdida"].mean():.1f}%')
    print(f'    % do texto perdido — max   : {trunc["pct_perdida"].max():.1f}%')
print('══════════════════════════════════════════')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# --- Histograma completo ---
ax = axes[0]
ax.hist(df['tokens_total'], bins=60, color='steelblue', edgecolor='white', linewidth=0.4)
ax.axvline(MAX_TOKENS, color='crimson', linewidth=2, linestyle='--', label=f'Limite {MAX_TOKENS} tokens')
ax.set_xlabel('Tokens por chunk (com prefixo "passage: ")')
ax.set_ylabel('Número de chunks')
ax.set_title('Distribuição de comprimento em tokens')
ax.legend()
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

# --- Zoom na cauda (chunks longos) ---
ax2 = axes[1]
limite_zoom = df['tokens_total'].quantile(0.90)
df_cauda = df[df['tokens_total'] > limite_zoom]
ax2.hist(df_cauda['tokens_total'], bins=40, color='darkorange', edgecolor='white', linewidth=0.4)
ax2.axvline(MAX_TOKENS, color='crimson', linewidth=2, linestyle='--', label=f'Limite {MAX_TOKENS} tokens')
ax2.set_xlabel('Tokens (zoom — 10% mais longos)')
ax2.set_ylabel('Número de chunks')
ax2.set_title(f'Cauda da distribuição (p90+ = {limite_zoom:.0f} tokens)')
ax2.legend()

plt.tight_layout()
plt.show()
print(f'Vermelho = corte em {MAX_TOKENS} tokens. Tudo à direita é truncado pelo modelo.')

## Passo 5 — O que é perdido na truncagem?

Para os chunks truncados, mostramos: o texto que o modelo **vê** e o que ele **não vê**.

In [ ]:
df_truncados = df[df['truncado']].sort_values('tokens_perdidos', ascending=False)

if df_truncados.empty:
    print('Nenhum chunk truncado — todos cabem em 512 tokens.')
else:
    print(f'{len(df_truncados):,} chunks são truncados. Exemplos (3 piores):\n')
    for _, row in df_truncados.head(3).iterrows():
        print(f'{'='*70}')
        print(f'Artigo  : {row["artigo"]}')
        print(f'Tokens  : {row["tokens_total"]} total  |  {row["tokens_perdidos"]} perdidos  |  {row["pct_perdida"]}% do texto ignorado')
        print(f'Chars   : {row["chars"]:,}')
        print()
        print('>>> TEXTO QUE O MODELO VÊ (início):')
        print(row['texto'][:300] + '…')
        print()
        print('>>> TEXTO QUE O MODELO NÃO VÊ (cortado):')
        perdido = row['texto_perdido']
        print(perdido[:300] + ('…' if len(perdido) > 300 else ''))
        print()

## Passo 6 — Top 20 chunks mais longos

In [ ]:
top20 = (
    df[['artigo', 'chars', 'tokens_total', 'tokens_perdidos', 'pct_perdida', 'truncado']]
    .sort_values('tokens_total', ascending=False)
    .head(20)
    .reset_index(drop=True)
)
top20.index += 1
top20.columns = ['Artigo', 'Chars', 'Tokens total', 'Tokens perdidos', '% perdida', 'Truncado']
top20

## Passo 7 — Faixas de comprimento

In [ ]:
faixas = [
    ('Muito curtos  (< 32 tokens)',  df['tokens_total'] < 32),
    ('Curtos        (32–128)',        (df['tokens_total'] >= 32)  & (df['tokens_total'] < 128)),
    ('Médios        (128–256)',       (df['tokens_total'] >= 128) & (df['tokens_total'] < 256)),
    ('Longos        (256–512)',       (df['tokens_total'] >= 256) & (df['tokens_total'] <= 512)),
    ('Truncados     (> 512)',         df['tokens_total'] > 512),
]

linhas = []
for label, mask in faixas:
    sub = df[mask]
    linhas.append({
        'Faixa': label,
        'Chunks': len(sub),
        '% do total': f'{len(sub)/len(df)*100:.1f}%',
        'Chars médios': f'{sub["chars"].mean():.0f}' if len(sub) else '—',
        'Tokens médios': f'{sub["tokens_total"].mean():.0f}' if len(sub) else '—',
    })

pd.DataFrame(linhas)

## Conclusão e Recomendações

Resumo do impacto da truncagem na qualidade dos embeddings.

In [ ]:
total = len(df)
truncados = df['truncado'].sum()

print('CONCLUSÃO')
print('='*60)
print()
print(f'  Chunks analisados : {total:,}')
print(f'  OK (≤ 512 tokens) : {total - truncados:,}  ({(total-truncados)/total*100:.1f}%)')
print(f'  Truncados         : {truncados:,}  ({truncados/total*100:.1f}%)')
print()

if truncados == 0:
    print('  ✅ Nenhum chunk é truncado — o chunking por artigo está adequado.')
elif truncados / total < 0.05:
    print('  ✅ Truncagem baixa (< 5%). A maioria dos artigos cabe em 512 tokens.')
    print('     Os truncados são provavelmente artigos com muitos incisos/parágrafos.')
    print()
    print('  Opções para os casos truncados:')
    print('    1. Sub-chunking: dividir artigos longos em incisos/parágrafos')
    print('    2. Aceitar a truncagem (conteúdo principal está geralmente no início)')
    print('    3. Usar um modelo com janela maior (ex: e5-mistral-7b-instruct, 4096 tokens)')
elif truncados / total < 0.20:
    print('  ⚠️  Truncagem moderada. Vale considerar sub-chunking dos artigos longos.')
    pct_media_perdida = df[df['truncado']]['pct_perdida'].mean()
    print(f'     Em média {pct_media_perdida:.1f}% do texto é ignorado nos chunks truncados.')
else:
    print('  ❌ Truncagem alta. O chunking por artigo gera muitos chunks grandes.')
    print('     Recomenda-se dividir por parágrafos ou incisos dentro dos artigos.')

print()
print('  Limite efetivo do modelo:')
print(f'    512 tokens totais')
print(f'    - 2 tokens especiais (CLS + SEP)')
n_tokens_prefixo = len(tokenizer(PREFIXO, add_special_tokens=False)["input_ids"])
print(f'    - {n_tokens_prefixo} tokens do prefixo "passage: "')
print(f'    = {512 - 2 - n_tokens_prefixo} tokens úteis de conteúdo por chunk')